# 03c — Demographics for the 2026 lines

Adds `year = 2026` to `acs.csv`, in the same schema, for the lines used in the 2026 House elections.

* **Unchanged states** (41 states, 262 districts): their 2026 lines are their 2024 lines, so the 2024 rows (2024 1-year ACS) are copied.
* **Redrawn states** (AL, CA, FL, LA, NC, OH, TN, TX, UT; 173 districts): Census hasn't tabulated the ACS on these maps, so each
  district is rebuilt the way `data-processing/house/nc_2020.py` rebuilds NC's 2020 map:
  1. **Map:** Census's 120th-Congress block equivalency file assigns every 2020 census block a 2026 district (`data/raw/baf_2026/`).
  2. **Weights:** each block group (and tract) is split across districts by where its 2020 Census (P.L. 94-171) population lives.
  3. **Estimates:** 2020–2024 ACS 5-year counts per block group (tract for mobility and nativity) are summed per district with
     those weights. Shares are ratios of the sums; medians are interpolated from the summed income and age distributions.
  4. **Vintage adjustment:** the ACS 1-year isn't published for areas this small, so the rebuild is a 5-year estimate. To match
     the 1-year rows, each state's published 2024 1-year minus 2020–2024 5-year gap is added (shares, household size) or applied
     as a ratio (medians). These rows are labelled `acs_vintage = "2024_1yr_rebuilt"`.

Assumption: people within a split block group look alike. Written to `data/interim/house/acs_2026.csv` and upserted into
`data/interim/house/acs.csv` (only the 2026 rows are replaced). Needs `CENSUS_API` in the environment.

Afterwards, rebuild the relative columns with `python data-processing/house/acs_supplement.py`, which compares 2026 against 2024 1-year
national and state values (`REFERENCE_YEAR` in `data-processing/house/acs_supplement.py`).

In [1]:
import hashlib
import json
import zipfile
from pathlib import Path

import pandas as pd

import acs
import nc_2020

YEAR = 2026
ACS_5YR = (2024, "acs5")  # 2020-2024 5-year: the newest release with block groups and tracts
ACS_1YR_YEAR = 2024  # acs.py's election year for the 2024 1-year, which the unchanged rows use
REBUILT_VINTAGE = "2024_1yr_rebuilt"

INTERIM_DIR = Path("../../data/interim")
HOUSE_INTERIM_DIR = INTERIM_DIR / "house"  # House-specific (district-level) interim data
BAF_DIR = Path("../../data/raw/baf_2026")
CD119_ZIP = Path("../../data/raw/cd119/cd119.zip")
# Census 119th-Congress block equivalency file (2024 lines), used only for the accuracy check; pinned like nc_2020.SOURCES
CD119_URL = "https://www2.census.gov/programs-surveys/decennial/rdo/mapping-files/2025/119-congressional-district-befs/cd119.zip"
CD119_SHA256 = "1433feb5178dc7b4188ee30f5f7f715851f4400740b8fe1ce606a876c6294bd6"

COUNTS = acs.normalize_counts(json.load(open("../../data/reference/district_counts.json")), YEAR)
FIPS = {abbr: fips for fips, abbr in acs.STATE_FIPS.items()}
KEY = acs.census_key()

# acs.py keeps 2026 = None (so `acs.py --all` skips it); register the 5-year here, in this session only, so its fetch and
# metadata checks resolve 2026 to the 2020-2024 5-year
acs.YEAR_TO_ACS_VINTAGE[YEAR] = ACS_5YR
acs.ACS_FIELDS[YEAR] = acs.vintage_fields(*ACS_5YR)
# nc_2020's block-group and tract formulas are built from the 2016-2020 5-year; they must be the 2024 5-year's too
FIVE_YEAR_FIELDS = {f: acs.ACS_FIELDS[YEAR][f] for f in ("white", "black", "hispanic", "college", "unemployment",
                                                          "moved_within_past_yr", "foreign_born", "housing_burden")}
if any(acs.ACS_FIELDS[2020][f] != spec for f, spec in FIVE_YEAR_FIELDS.items()):
    raise ValueError("2020 and 2024 5-year formulas differ; nc_2020's specs can't be reused")

# redrawn = any district whose PVI moved between Cook's 2025 (2024 lines) and 2026 releases, as in 03b
cook = pd.read_csv(HOUSE_INTERIM_DIR / "cook_pvi_public_2026_cleaned.csv")
REDRAWN = sorted(cook.loc[cook["2025 PVI"] != cook["2026 PVI"], "state"].unique())
baf_states = sorted(p.stem for p in BAF_DIR.glob("*.csv"))
if baf_states != REDRAWN:
    raise ValueError(f"Block files {baf_states} don't match the redrawn states {REDRAWN}")
print(f"Redrawn: {REDRAWN} ({sum(COUNTS[s] for s in REDRAWN)} districts)")

Redrawn: ['AL', 'CA', 'FL', 'LA', 'NC', 'OH', 'TN', 'TX', 'UT'] (173 districts)


## Plans and populations

In [2]:
def read_plan(path: Path, state: str) -> pd.Series:
    """2020 block GEOID -> district for one state's block file, checked against the state and its seat count."""
    plan = pd.read_csv(path, dtype=str, usecols=[0, 1])
    plan.columns = ["block", "district"]
    plan["district"] = plan["district"].astype(int)
    n = COUNTS[state]
    expected = {0} if n == 1 else set(range(1, n + 1))
    if plan["block"].duplicated().any() or not plan["block"].str.fullmatch(FIPS[state] + r"\d{13}").all():
        raise ValueError(f"{path}: block IDs must be unique 15-digit {state} GEOIDs")
    if set(plan["district"]) != expected:
        raise ValueError(f"{path}: districts {sorted(set(plan['district']))}, expected {sorted(expected)}")
    return plan.set_index("block")["district"]


def read_cd119(state: str) -> pd.Series:
    """The state's 119th-Congress (2024) plan from Census's national block equivalency file, downloaded and pinned."""
    if not CD119_ZIP.exists():
        CD119_ZIP.parent.mkdir(parents=True, exist_ok=True)
        nc_2020.urllib.request.urlretrieve(CD119_URL, CD119_ZIP)
    digest = hashlib.sha256(CD119_ZIP.read_bytes()).hexdigest()
    if digest != CD119_SHA256:
        raise ValueError(f"{CD119_ZIP} sha256 {digest} does not match the pinned {CD119_SHA256}; delete it to re-download")
    with zipfile.ZipFile(CD119_ZIP) as z, z.open("NationalCD119.txt") as f:
        plan = pd.read_csv(f, dtype=str)
    plan = plan[plan["GEOID"].str.startswith(FIPS[state])]
    return pd.Series(plan["CDFP"].astype(int).to_numpy(), index=plan["GEOID"].to_numpy())


def block_population(state: str) -> pd.Series:
    """2020 Census (P.L. 94-171) population of every block in the state, by 15-digit GEOID."""
    params = {"get": "P1_001N", "for": "block:*", "in": f"state:{FIPS[state]} county:* tract:*", "key": KEY}
    rows = acs._get_json(f"{acs.API_ROOT}/2020/dec/pl?{nc_2020.urllib.parse.urlencode(params)}")
    df = pd.DataFrame(rows[1:], columns=rows[0])
    return pd.Series(pd.to_numeric(df["P1_001N"]).to_numpy(), index=(df["state"] + df["county"] + df["tract"] + df["block"]))


def plan_shares(plan: pd.Series, population: pd.Series) -> pd.DataFrame:
    """Whole-block assignment in nc_2020.unit_weights' format; raises unless the plan covers exactly the state's blocks.

    Every block is in one district (share 1). Area is set to 1, so a block group with no 2020 population splits by block count.
    """
    if set(plan.index) != set(population.index):
        raise ValueError(f"Plan and census blocks differ: {len(set(population.index) - set(plan.index))} unassigned, "
                         f"{len(set(plan.index) - set(population.index))} unknown")
    return pd.DataFrame({"block": plan.index, "district": plan.to_numpy(), "area": 1.0, "share": 1.0})


plans = {st: read_plan(BAF_DIR / f"{st}.csv", st) for st in REDRAWN}
populations = {st: block_population(st) for st in REDRAWN}
balance = pd.DataFrame([
    {"state": st, "districts": COUNTS[st], "population": populations[st].sum(),
     "max_deviation_pct": 100 * (lambda p: (p / p.mean() - 1).abs().max())(populations[st].groupby(plans[st]).sum())}
    for st in REDRAWN
]).set_index("state")
balance  # 2026 plans are equal-population maps on 2020 counts, so deviations should be tiny

,districts,population,max_deviation_pct
state,,,
AL,7,5024279,0.000119
CA,52,39538223,2.340310
FL,28,21538187,0.000125
LA,6,4657757,0.008008
NC,14,10439388,0.000077
OH,15,11799448,0.000110
TN,9,6910840,0.000116
TX,38,29145505,0.000127
UT,4,3271616,0.000000


## ACS 5-year block groups and tracts

In [3]:
BLOCK_GROUP_VARS = (acs.spec_variables(nc_2020.BLOCK_GROUP_SPECS) + list(nc_2020.INCOME_CELLS)
                    + [v for cells in nc_2020.AGE_CELLS.values() for v in cells])
TRACT_VARS = acs.spec_variables(nc_2020.TRACT_SPECS)
acs.verify_variables(YEAR, BLOCK_GROUP_VARS + TRACT_VARS, {**acs.EXPECTED_LABELS, **nc_2020.REBUILD_LABELS},
                     {**acs.TABLE_CONCEPTS, **nc_2020.REBUILD_CONCEPTS})


def fetch_units(state: str, unit: str, variables: list[str]) -> pd.DataFrame:
    """2020-2024 ACS 5-year estimates for every block group or tract in the state, indexed by unit GEOID."""
    geo = ({"for": "block group:*", "in": f"state:{FIPS[state]} county:* tract:*"} if unit == "block group"
           else {"for": "tract:*", "in": f"state:{FIPS[state]} county:*"})
    raw = acs.fetch_acs(YEAR, KEY, geo, variables)
    ids = raw[[c for c in ("state", "county", "tract", "block group") if c in raw]].agg("".join, axis=1)
    return acs.parse_estimates(raw, variables, ids).set_axis(ids)


block_groups = {st: fetch_units(st, "block group", BLOCK_GROUP_VARS) for st in REDRAWN}
tracts = {st: fetch_units(st, "tract", TRACT_VARS) for st in REDRAWN}
pd.DataFrame({"block_groups": {st: len(v) for st, v in block_groups.items()}, "tracts": {st: len(v) for st, v in tracts.items()}})

,block_groups,tracts
AL,3925,1437
CA,25607,9129
FL,13388,5160
LA,4294,1388
NC,7111,2672
OH,9472,3168
TN,4562,1701
TX,18638,6896
UT,2020,716


In [4]:
def rebuild(state: str, plan: pd.Series) -> pd.DataFrame:
    """Unrounded 5-year schema fields (index = district) for one state's plan, from its block groups and tracts."""
    shares = plan_shares(plan, populations[state])
    bg = nc_2020.aggregate(block_groups[state], nc_2020.unit_weights(shares, populations[state], 12))
    tr = nc_2020.aggregate(tracts[state], nc_2020.unit_weights(shares, populations[state], 11))
    ids = pd.Series(bg.index.astype(str), index=bg.index)
    out = pd.concat([acs.compute_fields(bg, nc_2020.BLOCK_GROUP_SPECS, ids),
                     acs.compute_fields(tr, nc_2020.TRACT_SPECS, ids)], axis=1)
    out["median_income"] = nc_2020.interpolated_median(
        bg[list(nc_2020.INCOME_CELLS)], [nc_2020._lower_bound(b) for b in nc_2020.INCOME_BINS], pareto=True)
    ages = bg[list(nc_2020.AGE_CELLS["Male"])].to_numpy() + bg[list(nc_2020.AGE_CELLS["Female"])].to_numpy()
    out["median_age"] = nc_2020.interpolated_median(ages, [nc_2020._lower_bound(b) for b in nc_2020.AGE_BINS])
    return out[acs.COLUMNS[4:]]


rebuilt_5yr = pd.concat({st: rebuild(st, plans[st]) for st in REDRAWN}, names=["state", "district"])
rebuilt_5yr.round(2)

white  black  hispanic  college  median_income  median_age  \
state district                                                               
AL    1         65.11  24.96      4.12    27.63       63763.38       40.63   
      2         49.30  40.28      5.03    24.54       56374.55       38.69   
      3         71.01  19.59      4.05    24.08       60283.45       39.83   
      4         80.63   6.68      8.45    21.35       61136.44       40.49   
      5         68.18  17.49      7.03    36.06       77270.28       39.59   
...               ...    ...       ...      ...            ...         ...   
TX    38        46.71  10.06     27.66    50.60      106085.18       37.11   
UT    1         61.06   1.99     25.17    37.11       87716.97       33.88   
      2         78.66   1.00     13.99    35.06       95335.85       31.96   
      3         78.69   0.56     13.53    37.44       82576.40       31.07   
      4         79.93   0.76     11.37    41.11      112815.98       30.80   

                moved_within_past_yr  foreign_born  unemployment  poverty  \
state district                                                              
AL    1                        10.70          3.44          4.68    14.48   
      2                        12.72          3.74          5.61    19.49   
      3                        11.20          3.35          4.88    16.83   
      4                        10.29          4.20          4.26    16.08   
      5                        12.32          5.02          3.52    11.26   
...                              ...           ...           ...      ...   
TX    38                       14.27         22.91          5.25     8.66   
UT    1                        15.11         15.69          4.13     9.58   
      2                        11.90          6.02          2.82     7.99   
      3                        16.75          7.19          3.93    11.02   
      4                        12.22          7.01          3.33     5.56   

                avg_household_size  housing_burden  
state district                                      
AL    1                       2.50           28.95  
      2                       2.44           29.29  
      3                       2.49           25.24  
      4                       2.56           21.85  
      5                       2.45           22.51  
...                            ...             ...  
TX    38                      2.72           32.35  
UT    1                       2.67           31.38  
      2                       3.03           26.06  
      3                       2.95           29.36  
      4                       3.26           24.86  

[173 rows x 12 columns]

### Accuracy check: the same method on the 2024 (119th) lines

Census does publish the 2020–2024 5-year on the 119th-Congress districts, so rebuilding those districts from the same block
groups and comparing with Census's own values measures the method's error (as `nc_2020.sanity_check` does for NC).

In [5]:
def published(geo: dict[str, str], year: int) -> pd.DataFrame:
    """Census's published values for a geography, computed with acs.py's formulas for that year's vintage, unrounded."""
    raw = acs.fetch_acs(year, KEY, geo)
    ids = raw[[c for c in raw.columns if c in ("state", "congressional district")]].agg("-".join, axis=1)
    values = acs.parse_estimates(raw, acs.field_variables(year), ids).set_axis(ids)
    return acs.compute_fields(values, acs.ACS_FIELDS[year], ids.set_axis(ids))[acs.COLUMNS[4:]]


rebuilt_119 = pd.concat({st: rebuild(st, read_cd119(st)) for st in REDRAWN}, names=["state", "district"])
pub_119 = published({"for": "congressional district:*", "in": "state:" + ",".join(FIPS[s] for s in REDRAWN)}, YEAR)
pub_119.index = pd.MultiIndex.from_tuples(
    [(acs.STATE_FIPS[s], int(d)) for s, d in (i.split("-") for i in pub_119.index)], names=["state", "district"])
error_119 = (rebuilt_119 - pub_119.loc[rebuilt_119.index]).abs()
pd.DataFrame({"mean_abs_error": error_119.mean(), "max_abs_error": error_119.max()}).T.round(3)

,white,black,hispanic,college,median_income,median_age,moved_within_past_yr,foreign_born,unemployment,poverty,avg_household_size,housing_burden
mean_abs_error,0.057,0.035,0.047,0.040,541.271,0.063,0.048,0.059,0.016,0.032,0.003,0.048
max_abs_error,0.473,0.271,0.598,0.264,3383.166,0.261,0.265,0.452,0.127,0.166,0.011,0.308


## Vintage adjustment to the 2024 1-year

In [6]:
ADDITIVE = acs.PCT_FIELDS + ["avg_household_size"]  # shares (pp) and household size: add the 1-year minus 5-year gap
MULTIPLICATIVE = ["median_income", "median_age"]  # medians: scale by the 1-year / 5-year ratio

state_geo = {"for": "state:" + ",".join(FIPS[s] for s in REDRAWN)}
state_1yr, state_5yr = published(state_geo, ACS_1YR_YEAR), published(state_geo, YEAR)
state_1yr.index = state_5yr.index = [acs.STATE_FIPS[i] for i in state_5yr.index]
adjustment = pd.concat([state_1yr[ADDITIVE] - state_5yr[ADDITIVE], state_1yr[MULTIPLICATIVE] / state_5yr[MULTIPLICATIVE]], axis=1)

st = rebuilt_5yr.index.get_level_values("state")
rebuilt = rebuilt_5yr.copy()
rebuilt[ADDITIVE] = rebuilt_5yr[ADDITIVE] + adjustment.loc[st, ADDITIVE].to_numpy()
rebuilt[MULTIPLICATIVE] = rebuilt_5yr[MULTIPLICATIVE] * adjustment.loc[st, MULTIPLICATIVE].to_numpy()
adjustment.round(3)  # per state: pp / household-size gaps, and median ratios

,white,black,hispanic,college,moved_within_past_yr,foreign_born,unemployment,poverty,housing_burden,avg_household_size,median_income,median_age
AL,-0.596,-0.530,0.366,1.484,-0.376,0.474,-0.468,-0.433,0.489,-0.05,1.042,1.008
CA,-1.202,-0.100,0.590,1.041,-0.433,0.672,-0.672,-0.204,0.658,-0.05,1.010,1.013
FL,-1.348,-0.337,1.274,1.690,-0.362,1.187,-0.270,-0.526,2.296,-0.01,1.042,1.002
LA,-0.586,-0.349,0.564,0.792,-0.848,0.335,-1.140,-0.238,0.540,-0.07,1.004,1.018
NC,-1.030,-0.523,0.679,1.488,-0.526,0.826,-0.732,-0.482,1.747,-0.05,1.022,1.008
OH,-0.877,-0.095,0.343,0.864,-0.185,0.294,-0.769,-0.615,0.922,-0.02,1.012,1.003
TN,-0.528,-0.563,0.434,1.360,-0.732,0.434,-0.449,-0.298,0.691,-0.03,1.035,1.005
TX,-1.341,0.068,0.553,1.340,-0.165,0.851,-0.433,-0.379,1.168,-0.01,1.016,1.008
UT,-1.630,0.095,1.002,1.437,-0.705,0.835,0.217,-0.173,1.795,-0.06,1.016,1.016


### Plausibility: districts that barely changed

Where a 2026 district holds at least 95% of one 2024 district's 2020 population (and vice versa), its adjusted values should
be close to that 2024 district's published 1-year values in `acs.csv`.

In [7]:
acs_all = pd.read_csv(HOUSE_INTERIM_DIR / "acs.csv", dtype={"state": str, "acs_vintage": str})
acs_2024 = acs_all[acs_all["year"] == ACS_1YR_YEAR].set_index(["state", "district"])[acs.COLUMNS[4:]]

matches = []
for state in REDRAWN:
    pop = populations[state]
    overlap = pd.DataFrame({"new": plans[state], "old": read_cd119(state).reindex(pop.index), "pop": pop}).groupby(["new", "old"])["pop"].sum()
    share_new = overlap / overlap.groupby(level="new").transform("sum")
    share_old = overlap / overlap.groupby(level="old").transform("sum")
    for (new, old) in overlap[(share_new >= 0.95) & (share_old >= 0.95)].index:
        matches.append((state, new, old))
print(f"{len(matches)} nearly unchanged districts")
new_idx = pd.MultiIndex.from_tuples([(s, n) for s, n, _ in matches], names=["state", "district"])
old_idx = pd.MultiIndex.from_tuples([(s, o) for s, _, o in matches], names=["state", "district"])
gap = (rebuilt.loc[new_idx].to_numpy() - acs_2024.loc[old_idx].to_numpy())
pd.DataFrame(gap, columns=acs.COLUMNS[4:]).abs().agg(["mean", "max"]).round(3)

45 nearly unchanged districts


,white,black,hispanic,college,median_income,median_age,moved_within_past_yr,foreign_born,unemployment,poverty,avg_household_size,housing_burden
mean,0.666,0.454,0.471,0.745,1336.793,0.336,0.521,0.576,0.493,0.642,0.026,0.812
max,2.111,2.092,1.622,2.144,9301.549,1.488,1.629,2.108,1.833,2.087,0.066,2.678


## Assemble, validate and write

In [8]:
unchanged = acs_all[(acs_all["year"] == ACS_1YR_YEAR) & ~acs_all["state"].isin(REDRAWN)].assign(year=YEAR)
fields = acs.round_fields(rebuilt).reset_index()
redrawn_rows = fields.assign(year=YEAR, acs_vintage=REBUILT_VINTAGE)[acs.COLUMNS]
acs_2026 = pd.concat([unchanged, redrawn_rows]).sort_values(["state", "district"], ignore_index=True)[acs.COLUMNS]

acs.validate(acs_2026, YEAR, COUNTS)
acs.write_csv_atomic(acs_2026, HOUSE_INTERIM_DIR / "acs_2026.csv")
combined = acs.update_csv(acs_2026, HOUSE_INTERIM_DIR / "acs.csv", YEAR)
print(f"acs.csv: {len(combined)} rows, years {sorted(combined['year'].unique().tolist())}")
acs_2026.groupby("acs_vintage").size()

acs.csv: 4785 rows, years [2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020, 2022, 2024, 2026]


acs_vintage
2024_1yr            262
2024_1yr_rebuilt    173
dtype: int64